# P2.12 · Small VAE on MNIST (reconstruction-loss choice, ELBO with reparameterization)

**ID:** P2.12 · **Target:** 35 min · **Plan week:** 4 · **Part:** P (PyTorch first principles)

**The drill.** Train a fully connected VAE (784 → 400 → z=20 → 400 → 784) on MNIST by maximizing the ELBO

$$\mathcal{L}(x) = \mathbb{E}_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)] - \mathrm{KL}\big(q_\phi(z\mid x)\,\|\,\mathcal{N}(0, I)\big),\qquad
q_\phi(z\mid x) = \mathcal{N}(\mu_\phi(x), \operatorname{diag}\sigma^2_\phi(x)),$$

with the reparameterization $z = \mu + \sigma \odot \varepsilon,\ \varepsilon \sim \mathcal{N}(0, I)$ and the closed-form KL
$\mathrm{KL} = -\tfrac12 \sum_j (1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)$ (Kingma & Welling 2013, Appendix B).

**Must include (from the plan):** reconstruction-loss choice, ELBO with reparameterization. Concretely:
Bernoulli likelihood (BCE on [0, 1] pixels, from **logits**) vs Gaussian likelihood (MSE / 2σ²), and what the
choice does to the balance with the KL. Closed-form KL **checked against `torch.distributions.kl_divergence`**, the
reparameterization checked (gradients reach μ and log σ², sample moments right), initial-loss check
(784 · ln 2), overfit-one-batch, a grid of reconstructions and of samples from the prior.

**Rules.** No docs, no LLM. MNIST is in `DATA` (`download=False`). Each solution cell keeps only its spec.

## Sources
- D. P. Kingma, M. Welling, *Auto-Encoding Variational Bayes*, ICLR 2014, arXiv:1312.6114 —
  https://arxiv.org/abs/1312.6114 (ELBO, reparameterization trick, Gaussian KL in Appendix B).
- C. Doersch, *Tutorial on Variational Autoencoders*, 2016, arXiv:1606.05908 — https://arxiv.org/abs/1606.05908
  (intuition; Gaussian decoder σ as the reconstruction/KL trade-off).
- pytorch/examples `vae/main.py` (architecture 784-400-20, Adam 1e-3, batch 128, BCE + KL summed), commit acc295d,
  BSD-3: https://github.com/pytorch/examples/blob/acc295dc7b90714f1bf47f06004fc19a7fe235c4/vae/main.py.
  Adapted: the decoder returns logits and uses `binary_cross_entropy_with_logits` (stable), and the loss is
  averaged over the batch.
- G. Loaiza-Ganem, J. P. Cunningham, *The continuous Bernoulli: fixing a pervasive error in variational
  autoencoders*, NeurIPS 2019, arXiv:1907.06845 — https://arxiv.org/abs/1907.06845 (BCE on non-binary pixels is
  not a normalized likelihood).
- S. R. Bowman et al., *Generating Sentences from a Continuous Space*, CoNLL 2016, arXiv:1511.06349 —
  https://arxiv.org/abs/1511.06349 (posterior collapse; KL annealing).
- `torch.distributions.kl_divergence` / `Normal` docs: https://docs.pytorch.org/docs/stable/distributions.html

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.distributions import Normal, kl_divergence
from torch.utils.data import DataLoader, TensorDataset

from prep import DATA, Session, check, check_close, seed_everything

s = Session("P2.12", target_min=35)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. DATA. torchvision.datasets.MNIST(DATA, train=True/False, download=False). `.data` uint8 -> float in [0, 1],
#    flattened (N, 784). Do NOT standardize (the Bernoulli decoder needs targets in [0, 1]).
#    Val = last 5000 of a seeded permutation of train (generator seed 0). DataLoaders: train batch 128 shuffle;
#    val/test batch 1000. (Labels are not used.)
# YOUR CODE HERE


In [ ]:
# 2. CLOSED-FORM KL. kl_std_normal(mu, logvar) -> (B,): KL( N(mu, exp(logvar)) || N(0, I) ) summed over latent dims:
#    -0.5 * sum(1 + logvar - mu^2 - exp(logvar), dim=-1).
# YOUR CODE HERE


In [ ]:
# 3. MODEL. `VAE(d_in=784, d_h=400, d_z=20)`:
#    encode(x) -> (mu, logvar): Linear(784,400) + ReLU, then two heads Linear(400,20) for mu and for logvar.
#    reparameterize(mu, logvar) -> z = mu + exp(0.5*logvar) * eps, eps = randn_like.
#    decode(z) -> LOGITS (B, 784): Linear(20,400) + ReLU + Linear(400,784). (No sigmoid: the loss takes logits.)
#    forward(x) -> (logits, mu, logvar). make_model() -> fresh VAE on `dev`.
# YOUR CODE HERE


In [ ]:
# 4. NEGATIVE ELBO. neg_elbo(logits, x, mu, logvar, recon="bce", sigma=1.0) -> (loss, recon_term, kl_term), all
#    per-example quantities SUMMED over pixels / latent dims and AVERAGED over the batch (python-level scalars are fine
#    as tensors). recon="bce": F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(-1)
#    (= -log Bernoulli(x | sigmoid(logits))). recon="gauss": ((sigmoid(logits) - x)^2).sum(-1) / (2 sigma^2)
#    (Gaussian NLL up to the constant 784 * log(sigma * sqrt(2 pi))).
# YOUR CODE HERE


In [ ]:
# 5. SANITY CHECKS.
#    (a) `init_recon`: BCE recon term of a fresh model on one val batch (expect ≈ 784 ln 2 ≈ 543: logits ≈ 0 -> p = 0.5
#        for every pixel, whatever x is).
#    (b) Reparameterization: for fixed mu = [1, -2], logvar = log([0.25, 4]) draw 100 000 samples ->
#        `z_mean`, `z_std` (expect mu and [0.5, 2]); and one backward of z.sum() must give non-None grads on
#        mu and logvar (`grads_ok`).
#    (c) Overfit one batch of 64 with a fresh model, Adam lr=1e-3, 1000 steps, BCE ELBO ->
#        `overfit_recon` (recon term at the end; expect far below init, e.g. < 0.3 * init_recon).
# YOUR CODE HERE


In [ ]:
# 6. TRAIN (Bernoulli decoder). `fit(recon, epochs, sigma=1.0) -> (model, history)`: torch.manual_seed(0), fresh model,
#    Adam lr=1e-3, per epoch log train neg-ELBO and val (neg-ELBO, recon, KL) via
#    `evaluate(model, dl, recon, sigma) -> dict(loss, recon, kl)` (eval mode, no_grad, weighted by batch size over the
#    whole set; z is still sampled, a 1-sample ELBO estimate). history keys: "train", "val".
#    vae_bce, hist_bce = fit("bce", epochs=10).
# YOUR CODE HERE


In [ ]:
# 7. RECONSTRUCTION-LOSS CHOICE. Train the same VAE with the Gaussian decoder, sigma = 1 ("plain MSE / 2"), 5 epochs:
#    vae_gauss, hist_gauss = fit("gauss", 5, sigma=1.0). Compare the val KL of both models (`kl_bce`, `kl_gauss`)
#    and say what happened (look at the scale of the recon term vs the KL term).
# YOUR CODE HERE


In [ ]:
# 8. TEST + GRIDS (test once). `test_metrics = evaluate(vae_bce, test_dl)`; print neg-ELBO in nats/example.
#    Figure: row 1 = 8 test images, row 2 = their reconstructions sigmoid(logits) by vae_bce; then 8x8 grids of
#    samples decode(z), z ~ N(0, I), for vae_bce and vae_gauss side by side (torchvision.utils.make_grid).
# YOUR CODE HERE


## Self-check
Expects: `X_train`, `kl_std_normal`, `VAE`/`make_model` (encode/reparameterize/decode/forward), `neg_elbo`,
`init_recon`, `z_mean`, `z_std`, `grads_ok`, `overfit_recon`, `vae_bce`, `kl_bce`, `kl_gauss`, `test_metrics`.

In [ ]:
torch.manual_seed(1)
mu_r, lv_r = torch.randn(16, 20) * 2, torch.randn(16, 20)
ref = kl_divergence(Normal(mu_r, torch.exp(0.5 * lv_r)), Normal(torch.zeros(20), torch.ones(20))).sum(-1)
check_close("closed-form KL == torch.distributions", kl_std_normal(mu_r, lv_r), ref, atol=1e-4)
check_close("KL(N(0,1) || N(0,1)) == 0", kl_std_normal(torch.zeros(3, 20), torch.zeros(3, 20)), torch.zeros(3))
check("pixels in [0, 1], not standardized", X_train.min().item() >= 0 and X_train.max().item() <= 1)
x_r, l_r = torch.rand(8, 784), torch.randn(8, 784)
_, rec_r, kl_r = neg_elbo(l_r, x_r, mu_r[:8], lv_r[:8])
check_close("BCE recon = per-example SUM over pixels, batch MEAN",
            rec_r, F.binary_cross_entropy(torch.sigmoid(l_r), x_r, reduction="sum") / 8, rtol=1e-4)
check_close("KL term = batch mean of per-example KL", kl_r, ref[:8].mean(), atol=1e-4)
_, rec_g, _ = neg_elbo(l_r, x_r, mu_r[:8], lv_r[:8], recon="gauss", sigma=0.5)
check_close("Gaussian recon = SSE / (2 sigma^2), batch mean", rec_g, ((torch.sigmoid(l_r) - x_r) ** 2).sum() / 8 / (2 * 0.25), rtol=1e-4)
check("decoder returns logits (no Sigmoid module)", not any(isinstance(m, nn.Sigmoid) for m in vae_bce.modules()))
check("initial recon ~ 784 ln 2", abs(init_recon - 784 * math.log(2)) < 0.1 * 784 * math.log(2), f"got {init_recon:.1f}")
check_close("reparameterized z mean", z_mean, [1.0, -2.0], atol=0.03)
check_close("reparameterized z std = exp(0.5 logvar)", z_std, [0.5, 2.0], atol=0.03)
check("gradients flow to mu and logvar", grads_ok)
check("can overfit one batch (recon << init)", overfit_recon < 0.3 * init_recon, f"got {overfit_recon:.1f}")
torch.manual_seed(0)
xb = X_val[:500].to(dev)
vae_bce.eval()
with torch.no_grad():
    lg, mu_b, lv_b = vae_bce(xb)
check("forward shapes", lg.shape == (500, 784) and mu_b.shape == lv_b.shape and mu_b.shape[0] == 500)
check("test neg-ELBO < 125 nats", test_metrics["loss"] < 125, f"got {test_metrics['loss']:.1f}")
check("Bernoulli VAE uses its latent (KL > 10 nats)", kl_bce > 10, f"got {kl_bce:.1f}")
check("Gaussian sigma=1 decoder nearly collapses the posterior (KL much smaller)", kl_gauss < 0.5 * kl_bce, f"{kl_gauss:.1f} vs {kl_bce:.1f}")

## Say it out loud (P3)
1. Derive the ELBO from log p(x) in two lines. Which term is the "reconstruction" and which the "regularizer"?
2. Why the reparameterization trick? What would the gradient estimator be without it, and why is it worse?
3. Derive the closed-form KL between N(μ, σ²) and N(0, 1) for one dimension.
4. BCE vs MSE for the reconstruction: which likelihood does each correspond to? MNIST pixels aren't binary: is BCE still a valid likelihood?
5. Why did the σ = 1 Gaussian decoder end up with a tiny KL and blurry, similar-looking samples? What σ would you use, and how would you set it?
6. Why should the decoder output logits, and why sum over pixels (not mean) inside the ELBO?
7. What is posterior collapse, and name two fixes.
8. Why log σ² as the encoder output instead of σ?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up